# MongoDB Manager Usage Examples

This notebook demonstrates a practical `MongoDB` workflow with `py-dockerdb`, focusing on fast local setup, predictable lifecycle control, and reproducible execution from a single Python interface. Instead of manually wiring container commands for each run, you configure once, start the service, validate connectivity, and then execute database operations in a consistent sequence.

If you are comparing engines, this pattern helps you isolate query behavior and schema differences without rewriting orchestration code. Official reference: https://www.mongodb.com/docs/.

## Table Of Contents

- [Table Of Contents](#table-of-contents)
- [Prerequisites](#prerequisites)
- [1. Setup](#1-setup)
  - [Import Dependencies](#import-dependencies)
- [2. Creating a MongoDB Instance](#2-creating-a-mongodb-instance)
- [3. Start the Database](#3-start-the-database)
- [4. Connect and Create Collections](#4-connect-and-create-collections)
  - [Creating Collections and Inserting Data](#creating-collections-and-inserting-data)
- [5. Querying Data](#5-querying-data)
- [6. Advanced MongoDB Operations](#6-advanced-mongodb-operations)
- [7. Working with MongoDB Schema Validation](#7-working-with-mongodb-schema-validation)
- [8. Clean Up](#8-clean-up)
- [9. Conclusion](#9-conclusion)
- [10. Conclusion](#10-conclusion)

## Prerequisites

No environment variables are required unless you intentionally override the defaults used in this notebook.

Additional prerequisites:
- Docker must be installed and running before you call `create_db()`.
- Install project dependencies (for example, `pip install -e ".[test]"` or `pip install py-dockerdb`).
- Run cells top-to-bottom so container lifecycle state remains consistent.

Provider documentation: https://www.mongodb.com/docs/
Typical setup guide: https://www.mongodb.com/docs/manual/installation/
Docker image setup reference: https://hub.docker.com/_/mongo

Example datasets you can use to populate this database:
- Atlas sample datasets (e.g., sample_mflix, sample_airbnb): https://www.mongodb.com/docs/atlas/sample-data/
- JSON/CSV examples for local import: https://github.com/mongodb/docs-assets

## 1. Setup

### Install Required Packages

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
!pip install py-dockerdb pymongo

### Import Dependencies

This subsection details `import dependencies` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
import uuid
from pathlib import Path
from bson import ObjectId
from docker_db.dbs.mongo_db import MongoDBConfig, MongoDB

## 2. Creating a MongoDB Instance

Let's create a temporary directory for our database files:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
import tempfile
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-mongodb-{uuid.uuid4().hex[:8]}"
init_script_path = Path("configs", "mongodb", "init.js")
init_script_path.exists()

Now, let's set up the MongoDB configuration:

This context is intentionally explicit so the next code cell is not a blind execution step and you can quickly diagnose issues if behavior differs from expectations.

In [ ]:
# Generate a unique container name
container_name = f"demo-mongodb-{uuid.uuid4().hex[:8]}"

# Create a configuration for our database
config = MongoDBConfig(
    user="demouser",
    password="demopass",
    database="demodb",
    root_username="admin",
    root_password="adminpass",
    project_name="demo",
    workdir=temp_dir,
    container_name=container_name,
    retries=20,
    delay=3, 
    nit_script=init_script_path,
    env_vars={"YourEnvVar": "TestEnvironment"})

# Initialize the database manager
db_manager = MongoDB(config)

## 3. Start the Database

We'll now create and start the database:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Create and start the database
db_manager.create_db()
print(f"Database started successfully in container '{container_name}'")
print(f"Connection details:")
print(f"  Host: {config.host}")
print(f"  Port: {config.port}")
print(f"  User: {config.user}")
print(f"  Database: {config.database}")

## 4. Connect and Create Collections

Now that our database is running, let's connect to it and create some collections:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Connect to the database
client = db_manager.connection
db = client[config.database]

print("Connected to MongoDB successfully.")
print(f"Available collections: {db.list_collection_names()}")

### Creating Collections and Inserting Data

This subsection details `creating collections and inserting data` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
# Create a users collection and insert some data
users_collection = db["users"]

# Sample user data
users_data = [
    {"_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e0"), "username": "alice", "email": "alice@example.com", "age": 28},
    {"_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e1"), "username": "bob", "email": "bob@example.com", "age": 32},
    {"_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e2"), "username": "charlie", "email": "charlie@example.com", "age": 25},
]

# Insert users
result = users_collection.insert_many(users_data)
print(f"Inserted users with IDs: {[str(id) for id in result.inserted_ids]}")
print("Users collection created successfully.")
print(f"Available collections: {db.list_collection_names()}")

This step executes code block `15` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
# Create a posts collection and insert some data
posts_collection = db["posts"]

# Sample post data
posts_data = [
    {
        "_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e3"),
        "user_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e0"),
        "title": "Alice's First Post",
        "content": "Hello world from Alice!",
        "created_at": "2025-05-20T09:30:00.000Z"
    },
    {
        "_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e4"),
        "user_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e0"),
        "title": "Alice's Second Post",
        "content": "Another post from Alice",
        "created_at": "2025-05-20T10:15:00.000Z"
    },
    {
        "_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e5"),
        "user_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e1"),
        "title": "Bob's Introduction",
        "content": "Hi, this is Bob!",
        "created_at": "2025-05-20T11:00:00.000Z"
    },
    {
        "_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e6"),
        "user_id": ObjectId("64a7b1c2d3e4f5a6b7c8d9e2"),
        "title": "Charlie's Notes",
        "content": "Some notes from Charlie",
        "created_at": "2025-05-20T12:45:00.000Z"
    }
]

# Insert posts
result = posts_collection.insert_many(posts_data)
print(f"Inserted posts with IDs: {[str(id) for id in result.inserted_ids]}")
print("Posts collection created successfully.")
print(f"Available collections: {db.list_collection_names()}")

## 5. Querying Data

Now let's perform some queries on our collections:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Query all users
print("All users:")
print("-------------------------")
for user in users_collection.find():
    print(f"Username: {user['username']}, Email: {user['email']}, Age: {user['age']}")

# Find a specific user by username
print("\nFind user by username:")
print("-------------------------")
bob = users_collection.find_one({"username": "bob"})
print(f"Found user: {bob}")

# Find users within an age range
print("\nFind users by age range:")
print("-------------------------")
young_users = users_collection.find({"age": {"$lt": 30}})
for user in young_users:
    print(f"Username: {user['username']}, Email: {user['email']}, Age: {user['age']}")

This step executes code block `18` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
# Find posts by a specific user
alice_id = ObjectId("64a7b1c2d3e4f5a6b7c8d9e0")
print(f"Posts by Alice (user_id: {alice_id}):")
print("-------------------------")
alice_posts = posts_collection.find({"user_id": alice_id})
for post in alice_posts:
    print(f"Title: {post['title']}")
    print(f"Content: {post['content']}")
    print(f"Created at: {post['created_at']}\n")

# Find posts with a specific word in the title
print(f"Posts with 'notes' in the title:")
print("-------------------------")
notes_posts = posts_collection.find({"title": {"$regex": "notes", "$options": "i"}})
for post in notes_posts:
    print(f"Title: {post['title']}")
    print(f"Content: {post['content']}")
    print(f"Created at: {post['created_at']}")

## 6. Advanced MongoDB Operations

Let's demonstrate some more advanced MongoDB operations like aggregation pipelines:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Aggregate to count posts by user
pipeline = [
    {
        "$lookup": {
            "from": "users",
            "localField": "user_id",
            "foreignField": "_id",
            "as": "user"
        }
    },
    {"$unwind": "$user"},
    {
        "$group": {
            "_id": "$user._id",
            "username": {"$first": "$user.username"},
            "post_count": {"$sum": 1}
        }
    },
    {"$sort": {"post_count": -1}}
]

print("Post count by user:")
print("-------------------------")
post_counts = posts_collection.aggregate(pipeline)
for result in post_counts:
    print(f"Username: {result['username']}, Post count: {result['post_count']}")

This step executes code block `21` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
# Create indexes for better query performance
print("Creating indexes...")
users_collection.create_index("username", unique=True)
users_collection.create_index("email", unique=True)
print("Indexes created successfully.")

# List available indexes
print("\nAvailable indexes on 'users' collection:")
print(users_collection.index_information().keys())

## 7. Working with MongoDB Schema Validation

MongoDB supports schema validation for documents. Let's create a new collection with validation rules:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Create a new collection with schema validation
db.create_collection(
    "products",
    validator={
        "$jsonSchema": {
            "bsonType": "object",
            "required": ["name", "price", "category"],
            "properties": {
                "name": {
                    "bsonType": "string",
                    "description": "must be a string and is required"
                },
                "price": {
                    "bsonType": "number",
                    "minimum": 0,
                    "description": "must be a non-negative number and is required"
                },
                "category": {
                    "bsonType": "string",
                    "enum": ["Electronics", "Books", "Clothing", "Food"],
                    "description": "must be a string from the enum and is required"
                },
                "tags": {
                    "bsonType": "array",
                    "items": {
                        "bsonType": "string"
                    }
                }
            }
        }
    }
)

print("Products collection with schema validation created successfully.")

# Insert a valid document
products_collection = db["products"]
valid_product = {
    "name": "Laptop",
    "price": 999.99,
    "category": "Electronics",
    "tags": ["portable", "high-performance"]
}

result = products_collection.insert_one(valid_product)
print("Inserted valid product.")

# Try to insert an invalid document
try:
    invalid_product = {
        "name": "Invalid Product",
        "price": -10,  # Invalid: negative price
        "category": "Invalid"  # Invalid: not in enum
    }
    products_collection.insert_one(invalid_product)
except Exception as e:
    print("Document validation failed, as expected.")

print(f"Available collections: {db.list_collection_names()}")

## 8. Clean Up

When you're done with the database, you can delete it:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Close the connection
print("Closing connection...")
client.close()

# Delete the database container
db_manager.delete_db(running_ok=True)
print(f"Database container '{container_name}' deleted")

# Clean up the temporary directory
import shutil
shutil.rmtree(temp_dir)
print(f"Temporary directory '{temp_dir}' removed")

## 9. Conclusion

This notebook demonstrated how to:

1. Configure and create a MongoDB database with `MongoDB`
2. Create collections and insert documents
3. Perform various queries and aggregations
4. Work with indexes and schema validation
5. Clean up the database when finished

The `MongoDB` manager provides a convenient way to spin up MongoDB instances in Docker containers for development, testing, or demonstration purposes.

For production use, align this tutorial flow with your team standards: credential handling, migration strategy, backup policy, and monitoring. When needed, start from the provider setup docs (https://www.mongodb.com/docs/manual/installation/) and validate against real datasets before benchmarking query performance.